## Assignment 2: Unsupervised Learning

Welcome to the second assignment of CS 541! In Assignment 1 there was always a label to fit. Here we take the labels away, and the two hard problems of unsupervised learning show up immediately: choosing a structure, and deciding whether the structure you found is real. This assignment must be done individually.

After this assignment, you should be able to:  
1. Carry out PCA by hand on a small dataset, and decide when to standardize features before running it.  
2. Implement Lloyd's algorithm for K-means from scratch in numpy, and validate it against scikit-learn.  
3. Choose between K-means, hierarchical clustering, and DBSCAN based on the shape of the data.  
4. Evaluate a clustering with internal metrics, with withheld labels, and for stability under reseeding.  

In CS541, any work generated by an AI shouldn't be included without declaration. If you include material generated by an AI, the level of AI use should be properly documented, and the actual tool should be noted (e.g., "I used Codex to proofread the codes and draft the analysis"). 

Everything in Parts 2 through 5 uses datasets that ship inside scikit-learn, so nothing is downloaded. Run the cell below first.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine, load_digits, make_moons, make_blobs
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
                             adjusted_rand_score, normalized_mutual_info_score)

### 1. Unsupervised learning by hand (10')
Before running anything, work these out with pencil and paper. Answer in the markdown cell under each question. Show your intermediate steps, not just the final number -- the steps are what is graded. Use the *population* convention for every covariance in this part (divide by $n$, not by $n-1$), matching the lecture.  
1.1 PCA on four points (3')  
1.2 One run of Lloyd's algorithm (3')  
1.3 Single vs. complete linkage (2')  
1.4 Two short conceptual questions (2')  

#### 1.1 PCA on four points (3')
A dataset has $n = 4$ samples and $p = 2$ features:

| Sample | $x_1$ | $x_2$ |
| --- | --- | --- |
| A | 1 | 3 |
| B | 3 | 1 |
| C | 5 | 3 |
| D | 7 | 5 |

Work through the PCA recipe from the lecture:  
(a) Compute the feature means, and write down the four centered points.  
(b) Compute the $2 \times 2$ covariance matrix $\Sigma$, dividing by $n = 4$.  
(c) Compute both eigenvalues of $\Sigma$ exactly, and give a *unit* eigenvector for each. Verify one of your eigenpairs by multiplying it back through $\Sigma$.  
(d) Report the explained variance ratio of each component, as an exact fraction and as a percentage.  
(e) Project sample D onto PC1 and give its single coordinate in the 1-D reduced space.  
(f) Now suppose feature $x_2$ had been recorded in units ten times larger, so A becomes $(1, 30)$, B becomes $(3, 10)$, and so on, with $x_1$ untouched. Recompute $\Sigma$ and report the explained variance ratio of PC1. Which feature does PC1 now point along? Then compute the two eigenvalues of the *correlation* matrix of the original data, and explain in one sentence what that tells you about standardizing.  

Hint for (c): for a symmetric $2 \times 2$ matrix, $\lambda = \frac{1}{2}\left(\mathrm{tr}\,\Sigma \pm \sqrt{(\Sigma_{11}-\Sigma_{22})^2 + 4\Sigma_{12}^2}\right)$. The numbers here are chosen so the square root is exact.

*Your answer:*

#### 1.2 One run of Lloyd's algorithm (3')
Six points in two dimensions:

| Point | $x$ | $y$ |
| --- | --- | --- |
| A | 0 | 0 |
| B | 2 | 0 |
| C | 3 | 1 |
| D | 8 | 0 |
| E | 9 | 1 |
| F | 10 | 0 |

Run K-means with $K = 2$ from a deliberately poor initialization: $c_1 = A = (0,0)$ and $c_2 = B = (2,0)$. Report *squared* Euclidean distances throughout. The $\arg\min$ is the same either way, and the squared distances here are exact, so no rounding creeps in.  
(a) Iteration 1: give the squared distance from every point to each centroid, the resulting assignment, and the two updated centroids.  
(b) Iteration 2: repeat. Name every point that changed cluster, and explain why the update step in (a) made that change possible.  
(c) Run the assign step once more. Has the algorithm converged? Report the final inertia.  
(d) Compute the silhouette coefficient of point A in the final clustering. Use *unsquared* Euclidean distances here, since the silhouette formula is defined on distances. Report $a(A)$, $b(A)$, and $\text{silhouette}(A)$ to two decimals.  
(e) Suppose instead we had used k-means++ and the first centroid drawn was A. Give the probability that each of the remaining five points is chosen as the second centroid, and hence the probability that the second centroid lands in the right-hand group $\{D, E, F\}$.  

*Your answer:*

#### 1.3 Single vs. complete linkage (2')
Five points on a number line: $A = 0$, $B = 4$, $C = 6$, $D = 7$, $E = 14$. Distance is absolute difference.  
(a) Write out the $5 \times 5$ distance matrix.  
(b) Run agglomerative clustering with **single** linkage. Give the merge sequence and the height of each of the four merges.  
(c) Repeat with **complete** linkage.  
(d) For each linkage, give the range of cut heights that yields 2 clusters, and the range that yields 3 clusters. Do the two linkages produce the *same* partition at $K = 2$? At $K = 3$? State precisely what the choice of linkage did and did not change on this dataset.  
(e) In one or two sentences: on a dataset shaped like two interleaved crescents, which of the two linkages would you expect to recover the crescents, and why?  

*Your answer:*

#### 1.4 Two short conceptual questions (2')
(a) A classmate has 40 samples. They run K-means for every $K$ from 1 to 40, plot inertia, observe that inertia is lowest at $K = 40$, and conclude that the data has 40 clusters. Identify the error, state what the inertia is at $K = 40$ and why, and name two things they should do instead.  
(b) A dataset has $n = 4$ samples and two features. Class 0 contains $(-3,\ 0.1)$ and $(3,\ 0.1)$; class 1 contains $(-3,\ -0.1)$ and $(3,\ -0.1)$. The class labels are *not* available to PCA. Compute $\Sigma$ and its explained variance ratios. If you reduce to $k = 1$ component, what happens to the class information? State the general lesson in one sentence.  

*Your answer:*

### 2. PCA on real data (8')
Now run the same recipe at scale, on two datasets that ship with scikit-learn. The **wine** dataset has 178 samples and 13 chemical measurements from 3 cultivars. The **digits** dataset has 1797 handwritten digits stored as 8x8 images, so 64 pixel features, across 10 classes. Ignore the class labels in this part entirely -- they come back in Part 5.  
2.1 Load both datasets. For wine, build a DataFrame named `wine_feature_stats`, indexed by feature name, with columns `mean` and `std`, and print it. Briefly comment on what the two columns tell you about running PCA on these features as they are.  
2.2 Implement `pca_eigen()` below, following the six-step PCA recipe from the lecture. Then validate it against `sklearn.decomposition.PCA` on the standardized wine data.  
2.3 Run `pca_eigen()` on wine twice, once with `standardize=False` and once with `standardize=True`. For each, report the explained variance ratio of PC1 and the three features with the largest absolute loading on PC1. Store the two full ratio arrays as `evr_wine_raw` and `evr_wine_std`. Compare the two results and comment on your observations.  
2.4 Produce a scree plot for the digits dataset: the explained variance ratio per component as bars, the cumulative ratio as a line on the same axes, and a horizontal reference line at 0.90. Store the ratio array as `evr_digits`, and store the smallest number of components whose cumulative ratio reaches 0.90 and 0.95 as `n_components_90` and `n_components_95`. Comment on where the elbow is, and on whether you would standardize the digits features before PCA. Justify that using the pixel scale, not a general rule.  

Following are some specifications that may be helpful:  
- Use the population convention for the covariance matrix, dividing by $n$, to match the lecture. scikit-learn divides by $n-1$ instead, so your *eigenvalues* will differ from `pca.explained_variance_` by a factor of $n/(n-1)$. The explained variance *ratios* are identical, because that factor cancels. Compare ratios.  
- Use `np.linalg.eigh`, not `np.linalg.eig`. A covariance matrix is symmetric, and `eigh` exploits that to return real eigenvalues in ascending order. `eig` can hand back a complex dtype and an arbitrary order.  
- An eigenvector and its negation describe the same axis, so the *sign* of a component is arbitrary. Never compare signs against scikit-learn; compare absolute values.  
- The loadings of a component are the entries of its eigenvector: entry $j$ of PC1 says how much original feature $j$ contributes to PC1.  

In [ ]:
# Starter
wine = load_wine()
X_wine, y_wine = wine.data, wine.target
digits = load_digits()
X_digits, y_digits = digits.data, digits.target
print("wine:", X_wine.shape, "  digits:", X_digits.shape)

# TODO -- 2.1 build wine_feature_stats (index = wine.feature_names, columns = mean, std) and print it
wine_feature_stats = pass


def standardize_features(X):
    """Zero mean, unit variance, column by column. Given -- do not modify."""
    mu = X.mean(axis=0)
    sd = X.std(axis=0)                   # population std, matching the lecture
    sd = np.where(sd == 0.0, 1.0, sd)    # a constant feature would divide by zero
    return (X - mu) / sd


def pca_eigen(X, standardize=False):
    """Return (eigvals, eigvecs, evr), all sorted by descending eigenvalue.
    eigvecs[:, i] is the i-th principal component."""
    # TODO -- step 1: standardize if asked, then center
    Xc = pass

    # TODO -- step 2: the covariance matrix, dividing by n (population convention)
    # Note: don't use np.cov's default, which divides by n-1. Pass bias=True, or write it out.
    Sigma = pass

    # TODO -- step 3: eigenvalues and eigenvectors of Sigma, via np.linalg.eigh
    eigvals, eigvecs = pass

    # TODO -- step 4: sort both by DESCENDING eigenvalue
    eigvals, eigvecs = pass

    # Steps 5 and 6 (choosing k, and projecting) are the caller's job below.
    # TODO -- the explained variance ratio of each component
    evr = pass
    return eigvals, eigvecs, evr


# TODO -- 2.2 validate pca_eigen against sklearn's PCA on the standardized wine data.
# Compare the explained variance ratios, and compare the PC1 loadings up to sign.
# Note: don't standardize here and again inside pca_eigen -- pick one place to do it.

# TODO -- 2.3 the standardize-or-not contrast, plus the top-3 PC1 loadings for each
evr_wine_raw = pass
evr_wine_std = pass

# TODO -- 2.4 the digits scree plot, then the two component counts
evr_digits = pass
n_components_90 = pass
n_components_95 = pass

*Your answers for 2.1, 2.3 and 2.4:*

### 3. K-means from scratch (8')
Implement Lloyd's algorithm yourself, check it against the library, and then use it to choose $K$.  
3.1 Complete `kmeans_fit()` below. It must return `(centroids, labels, inertia, n_iters)`.  
3.2 Validate. Run your `kmeans_fit(X_digits, K=10, seed=0)` and `KMeans(n_clusters=10, n_init=10, random_state=0)` on the same data. Store the two inertias as `my_inertia_k10` and `sklearn_inertia_k10`, report their relative difference, and report the adjusted Rand index between the two label vectors. Briefly comment: should these two numbers be identical, and if not, why not?  
3.3 Sweep $K$ from 2 to 15 on the digits data using *your* implementation. Build two dicts mapping $K$ to a number, `inertia_by_k` and `silhouette_by_k`, and plot them side by side in two subplots. Store the $K$ that maximizes the average silhouette as `best_k_silhouette`, and the $K$ you would read off the elbow as `best_k_elbow`.  
3.4 The digits data has 10 true classes. Compare `best_k_elbow`, `best_k_silhouette`, and 10, and comment on your observations. Explain why inertia alone could never have told you the answer.  

Following are some specifications that may be helpful:  
- Initialize each restart by sampling $K$ *distinct* rows of `X` as the starting centroids, using the `rng` that is already created for you so the run is reproducible.  
- Lloyd's algorithm converges to a *local* optimum that depends on the initialization, so `kmeans_fit` runs `n_init` independent restarts and keeps the one with the lowest inertia. This matters for 3.3: with a single restart, the inertia curve over $K$ is not even guaranteed to decrease, and the elbow becomes unreadable.  
- Assign each point with `np.argmin`, which breaks ties toward the lowest index. That keeps a seeded run reproducible across machines.  
- Stop a restart when no label changes, or when the largest centroid movement falls below `tol`, or after `max_iters` passes, whichever comes first.  
- Inertia is the *total* squared distance from every point to its own centroid, matching the definition in lecture. Report the total, not the mean.  
- Note: don't call `sklearn.cluster.KMeans` anywhere inside `kmeans_fit`, and don't call it inside the 3.3 sweep either. The point of this part is your own loop.  

In [ ]:
# Starter
def pairwise_sq_dists(X, C):
    """Squared Euclidean distances between every row of X (n, p) and every row
    of C (K, p). Returns an (n, K) array. Given -- do not modify."""
    return ((X[:, None, :] - C[None, :, :]) ** 2).sum(axis=-1)


def _kmeans_one_run(X, K, rng, max_iters, tol):
    """One restart of Lloyd's algorithm. Returns (centroids, labels, inertia, n_iters)."""
    n = X.shape[0]
    centroids = X[rng.choice(n, size=K, replace=False)].astype(float).copy()
    labels = np.full(n, -1)

    for it in range(max_iters):
        # TODO -- assign step: the squared distances, then the nearest centroid per point
        d2 = pass
        new_labels = pass

        # TODO -- update step: move every centroid to the mean of its members.
        # The empty-cluster branch is given, because np.mean of an empty slice is nan.
        new_centroids = centroids.copy()
        for k in range(K):
            members = X[new_labels == k]
            if len(members) == 0:
                new_centroids[k] = X[np.argmax(d2.min(axis=1))]  # worst-served point
            else:
                new_centroids[k] = pass  # TODO

        shift = np.abs(new_centroids - centroids).max()
        converged = (new_labels == labels).all() or shift < tol
        centroids, labels = new_centroids, new_labels
        if converged:
            break

    # TODO -- inertia: the total squared distance from each point to its OWN centroid
    inertia = pass
    return centroids, labels, inertia, it + 1


def kmeans_fit(X, K, seed=0, n_init=10, max_iters=100, tol=1e-6):
    """Run n_init restarts of Lloyd's algorithm and keep the lowest-inertia one."""
    rng = np.random.default_rng(seed)
    best = None
    for _ in range(n_init):
        # TODO -- one restart, then keep it if its inertia beats the best so far
        pass
    return best


# TODO -- 3.2 validate against sklearn, and print the relative difference and the ARI
my_inertia_k10 = pass
sklearn_inertia_k10 = pass

# TODO -- 3.3 sweep K from 2 to 15, fill both dicts, then plot two subplots
inertia_by_k = pass
silhouette_by_k = pass
best_k_elbow = None
best_k_silhouette = None

*Your answers for 3.2 and 3.4:*

### 4. Where K-means fails (7')
K-means is not wrong, it is *assuming* something. Here we build two datasets that violate its assumptions in two different ways, and compare it against hierarchical clustering and DBSCAN.  
4.1 Generate the two datasets with the exact calls given in the starter, and plot both, coloured by their true labels. `moons` is two interleaved crescents. `blobs` is three spherical clusters whose standard deviations differ by a factor of five.  
4.2 On each dataset, run five methods: K-means with the correct $K$; agglomerative clustering with `single`, `complete`, and `average` linkage; and DBSCAN. Fill a DataFrame named `comparison_df` with exactly the columns `["dataset", "method", "n_clusters_found", "ari", "silhouette"]` and print it. Plot the ten clusterings in a 2 x 5 grid using the given `plot_clusters()` helper.  
4.3 DBSCAN's answer depends entirely on $\varepsilon$. For the moons dataset, sweep $\varepsilon$ over the grid `EPS_GRID` given in the starter at `min_samples=5`, and fill a DataFrame named `dbscan_grid_df` with exactly the columns `["eps", "n_clusters_found", "noise_frac", "ari"]`. Report the $\varepsilon$ you chose and how you chose it.  
4.4 Write your analysis. Address all four:  
- Which methods recover the crescents, and which do not? Explain *geometrically* why K-means cannot, referring to the shape of the region each centroid owns.  
- Single and complete linkage disagree sharply on `moons`, and they disagree in the opposite direction on `blobs`. Connect both to your answer in 1.3(e).  
- On `moons`, compare the silhouette score of the *true* labels against the silhouette score of the K-means labels. Which is higher? What does that tell you about using an internal metric to choose between algorithms?  
- On the unequal-variance `blobs`, DBSCAN has a harder time than it did on `moons`. Explain why a single global $\varepsilon$ is the problem.  

Following are some specifications that may be helpful:  
- Note: don't use the true labels to pick $\varepsilon$ or $K$. Choose them from the data, then report the ARI afterwards. The ARI column exists so *you* can see how well the label-free choice did, and that is only an honest exercise if you make the choice first.  
- DBSCAN labels noise as `-1`, which is not a cluster. Both given helpers already account for that.  

In [ ]:
# Starter
X_moons, y_moons = make_moons(n_samples=500, noise=0.06, random_state=0)
X_blobs, y_blobs = make_blobs(n_samples=500,
                              centers=[[-6.0, 0.0], [0.0, 0.0], [6.0, 0.0]],
                              cluster_std=[1.0, 2.5, 0.5], random_state=0)
DATASETS = {"moons": (X_moons, y_moons, 2), "blobs": (X_blobs, y_blobs, 3)}
EPS_GRID = [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.60]
METHODS = ["kmeans", "agglo-single", "agglo-complete", "agglo-average", "dbscan"]


def plot_clusters(X, labels, title, ax):
    """Scatter X coloured by labels; DBSCAN noise (-1) is drawn as a grey x.
    Given -- do not modify."""
    noise = labels == -1
    ax.scatter(X[~noise, 0], X[~noise, 1], c=labels[~noise], s=8, cmap="tab10")
    ax.scatter(X[noise, 0], X[noise, 1], c="lightgray", s=8, marker="x")
    ax.set_title(title, fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])


def n_clusters_found(labels):
    """How many real clusters, excluding DBSCAN's noise label. Given."""
    return len(set(np.asarray(labels).tolist()) - {-1})


def safe_silhouette(X, labels):
    """Average silhouette over the non-noise points. Returns nan when fewer than
    two clusters survive -- silhouette_score raises on a single-cluster labeling,
    and DBSCAN produces one easily. Given -- do not modify."""
    labels = np.asarray(labels)
    mask = labels != -1
    if len(set(labels[mask].tolist())) < 2 or mask.sum() < 3:
        return float("nan")
    return float(silhouette_score(X[mask], labels[mask]))


# TODO -- 4.1 plot both datasets coloured by their true labels

# TODO -- 4.3 the eps sweep on moons (do this before 4.2, so it informs your choice)
dbscan_grid_df = pass

# TODO -- 4.2 pick an eps per dataset, run the five methods on each, fill comparison_df,
# and plot the 2x5 grid. METHODS above fixes the column order for the grid.
DBSCAN_PARAMS = {"moons": dict(eps=None, min_samples=5),
                 "blobs": dict(eps=None, min_samples=5)}
comparison_df = pass

*Your answers for 4.3 and 4.4:*

### 5. Reduce, then cluster, and evaluate (10')
This is the pipeline from the lecture, end to end, on the digits dataset. The true labels `y_digits` are **withheld from every clustering call** and used only for evaluation afterwards. Treat that as a rule, not a suggestion.  
5.1 Complete `reduce_then_cluster()` below: reduce `X` to `n_components` with PCA, then cluster the reduced representation with K-means. `n_components=None` means skip the reduction and cluster the raw 64 features.  
5.2 For `n_components` in `N_COMPONENTS_GRID` with $K = 10$ and `seed=0`, build a DataFrame named `eval_df` with exactly the columns `["n_components", "ari", "nmi", "silhouette", "davies_bouldin"]`, and print it. Store the `n_components` with the highest ARI as `best_n_components`. Plot ARI and average silhouette against `n_components` on the same axes with two y-scales.  
5.3 Stability. Fix `n_components = best_n_components` and $K = 10$, and repeat the clustering five times with the seeds in `SEEDS`. Record the five ARI-against-truth numbers in a list named `stability_ari`, and report their mean and standard deviation. Then compute the adjusted Rand index between every *pair* of those five clusterings -- ten pairs, no labels involved -- and store the mean as `stability_pairwise_ari`.  
5.4 Visualize the ten K-means centroids from your best pipeline as 8x8 images, by mapping them back into pixel space with the fitted PCA's `inverse_transform`. Plot them in a 2 x 5 grid.  
5.5 Write your analysis. Address all four:  
- Does reducing before clustering help? At which `n_components` does ARI peak, and what is happening at the two ends of the range?  
- Average silhouette and ARI do not peak at the same `n_components`. Describe the disagreement and explain it. Which of the two would you have had access to if the labels genuinely did not exist?  
- What do `stability_ari` and `stability_pairwise_ari` tell you that a single run cannot? Which of the two could you still compute with no labels at all?  
- Look at your ten centroid images. Name one digit that clearly got its own centroid and one pair of digits that appear to have been merged or split, and say in one sentence why you would not report these ten clusters to someone as "the ten digits".  

Following are some specifications that may be helpful:  
- Note: don't standardize the digits features here. All 64 are pixel intensities on the same 0-16 scale, and 3 border pixels are constant zero, so dividing by their standard deviation is either a division by zero or an amplification of pure noise.  
- Fit the PCA on `X` and cluster the transformed array. Do not refit the PCA once per seed in 5.3; only the K-means seed changes there.  
- Cluster IDs are arbitrary names, so never compare them to class IDs with accuracy -- a relabeled clustering is the same clustering. The adjusted Rand index and normalized mutual information are invariant to relabeling, which is exactly why we use them.  
- Compute `silhouette` and `davies_bouldin` in the space you clustered in, and say which space that was. They are not comparable across different `n_components` for free; mention this in your analysis.  
- Lower Davies-Bouldin is better. Higher silhouette, ARI, and NMI are better.  

In [ ]:
# Starter
N_COMPONENTS_GRID = [None, 2, 5, 10, 20, 40]
SEEDS = [1, 2, 3, 31, 42]


def reduce_then_cluster(X, n_components, K, seed):
    """Return (labels, X_reduced, pca). pca is None when n_components is None."""
    if n_components is None:
        X_reduced, pca = X, None
    else:
        # TODO -- fit a PCA with n_components on X, then transform X
        pca = pass
        X_reduced = pass

    # TODO -- K-means on X_reduced. Use n_init=10 and random_state=seed.
    labels = pass
    return labels, X_reduced, pca


# TODO -- 5.2 build eval_df over N_COMPONENTS_GRID, then best_n_components, then the two-axis plot
eval_df = pass
best_n_components = None

# TODO -- 5.3 stability across the five SEEDS, against truth and pairwise
stability_ari = pass
stability_pairwise_ari = pass

# TODO -- 5.4 the ten centroids as 8x8 images, via pca.inverse_transform

*Your answer for 5.5:*